In [5]:
from langchain_community.document_loaders import PyPDFLoader

import os
from dotenv import load_dotenv
load_dotenv()

PDF_PATH = "/home/soni/Desktop/learning-in-public/Projects/data/Enabl3_Interview_Prep.pdf"
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages of pdf.")
print(pages[0].page_content[:500])

Loaded 11 pages of pdf.
1:  Tell  Us  About  your  self:  
 
Hello  sir/mam.  My  name  is  Vishal  Soni,  and  I’m  currently  pursuing  my  Bachelor  of  
Engineering
 
in
 
Information
 
Technology
 
from
 
IIIT
 
Pune.
 
I’m  a  systems-focused  developer  with  a  strong  interest  in  backend  engineering,  
workflow-driven
 
systems,
 
automation,
 
and
 
solving
 
operational
 
problems
 
through
 
technology.
 
I’m  currently  working  as  a  Full  Stack  Developer  Intern  at  Attento  Technologies,  where  I


## 1: Chunking

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
  chunk_size=600,
  chunk_overlap=100,
  separators=["\n\n", "\n", ".", " "],
)

chunks = splitter.split_documents(pages)
# len(chunks)
len(chunks[0].page_content)

588

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)
print(vector_store)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved)} chunks:\n")
for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10158.33it/s]


Query: What is VoLTE and how does it improve call quality?
Retrieved 3 chunks:

--- Chunk 1 ---
maintainability,
 
readability,
 
and
 
developer
 
productivity
 
through
 
better
 
IDE
 
support
 
and
 
type
 
safety.
 
During  my  internship,  TypeScript  was  especially  useful  while  handling  complex  
backend
 
request
 
and
 
response
 
structures
 
in
 
NestJS
 
APIs.
 
Strong
 
typin

--- Chunk 2 ---
maintainability,
 
readability,
 
and
 
developer
 
productivity
 
through
 
better
 
IDE
 
support
 
and
 
type
 
safety.
 
During  my  internship,  TypeScript  was  especially  useful  while  handling  complex  
backend
 
request
 
and
 
response
 
structures
 
in
 
NestJS
 
APIs.
 
Strong
 
typin

--- Chunk 3 ---
maintainability,
 
readability,
 
and
 
developer
 
productivity
 
through
 
better
 
IDE
 
support
 
and
 
type
 
safety.
 
During  my  internship,  TypeScript  was  especially  useful  while  handling  complex  
backend
 
request
 
and
 
response
 
structures
 
in
 

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

# --- Helper: join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


# --- System prompt: ground the LLM in the retrieved context ---
SYSTEM_PROMPT = """\
You are a helpful chat assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# --- LLM via Groq API ---
# llm = ChatGroq(
#     model="qwen/qwen3-32b",
#     temperature=0,
#     reasoning_format="parsed",
# )

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)


# --- Assemble the chain ---
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled.")

RAG chain assembled.


In [20]:
question = "go fuck your self you dick"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: go fuck your self you dick

A: I'm here to provide helpful and respectful assistance. The context you provided earlier was about Vishal Soni, a systems-focused developer. If you have any questions or need information related to that context, I'll be happy to help. However, I won't engage in or respond to abusive language. Is there something else I can assist you with?
